# AI Clip Studio v2 — Urdu + English, Auto + Custom + Text Editor

This notebook is the **cloud processing engine** for the included Android-friendly `index.html`. It handles videos you have rights to edit and publish. Its web preview is a **local clip planner**, not a hosted rendering service.

**First run:** Upload a 3–5 minute MP4 to `MyDrive/AI-Clip-Studio/INPUT` → open this notebook in Google Colab → **Runtime → Run all** → authorize Google Drive → download rendered clips from `OUTPUT/<run>/`.

**Automatic highlights:** `MODE = auto`. **Exact cuts:** create `clip_plan.json` in `index.html`, upload it alongside the same video, and choose `MODE = manual` or `both`. **Text-based editing:** import `full_transcript.json` from the first run into `index.html`; choose lines and correct captions; save new `clip_plan.json`; upload it and rerun with `MODE = manual` or `both`.

**Supported speech preferences:** Urdu (`ur`), English (`en`) or automatic language detection. Roman Urdu and mixed-language dialogue can be inconsistent with lightweight models; check every subtitle. Editing transcription text changes captions, **not** the original spoken audio.

**Practical constraints:** 8-minute audio chunks reduce peak memory, but no upload is literally unlimited. Long videos and high-volume usage can exceed Google Colab's free compute quota/session lifespan or Google Drive space. Someone must still **Run all**; no unattended server or direct social posting is included. Copyright permission is required regardless of cropping or captions. See `GUIDE.html` for setup and troubleshooting.


In [ ]:
#@title 1) Install free video and multilingual transcription tools
!apt-get -qq update >/dev/null
!apt-get -qq install -y ffmpeg fonts-noto-core >/dev/null
!pip -q install 'faster-whisper>=1.0,<2'
print('Ready: FFmpeg and multilingual faster-whisper installed.')


In [ ]:
#@title 2) Connect Google Drive and set preferences
from google.colab import drive
drive.mount('/content/drive')

MODE = "auto" #@param ["auto", "manual", "both"]
SOURCE_FILENAME = "" #@param {type:"string"}
SPEECH_LANGUAGE = "auto" #@param ["auto", "ur", "en"]
MODEL_SIZE = "base" #@param ["tiny", "base", "small"]
PLATFORM_PRESET = "All platforms" #@param ["All platforms", "Instagram Reels", "TikTok", "YouTube Shorts"]
AUTO_TARGET_SECONDS = 0 #@param {type:"slider", min:0, max:180, step:5}
CLIPS_COUNT = 3 #@param {type:"slider", min:1, max:5, step:1}
CHUNK_MINUTES = 8 #@param {type:"slider", min:2, max:15, step:1}
# 0 = use editorial target preset (not an official platform requirement).
# Use manual cuts to keep exact custom start/end points; those are NOT auto-shortened.

from pathlib import Path
ROOT = Path('/content/drive/MyDrive/AI-Clip-Studio')
INPUT = ROOT / 'INPUT'
OUTPUT = ROOT / 'OUTPUT'
INPUT.mkdir(parents=True, exist_ok=True)
OUTPUT.mkdir(parents=True, exist_ok=True)
print('INPUT:', INPUT)
print('OUTPUT:', OUTPUT)


In [ ]:
#@title 3) Urdu/English transcription, highlight selection, and export helpers
import json, math, re, shutil, subprocess, time
from bisect import bisect_right, bisect_left
from pathlib import Path
from faster_whisper import WhisperModel

WORK=Path('/content/clipstudio_work'); WORK.mkdir(parents=True,exist_ok=True)
VIDEO_EXTS={'.mp4','.mov','.mkv','.m4v','.webm'}
PRESET_TARGETS={'All platforms':35,'Instagram Reels':30,'TikTok':40,'YouTube Shorts':45}
TARGET_SECONDS = AUTO_TARGET_SECONDS or PRESET_TARGETS.get(PLATFORM_PRESET,35)
HIGHLIGHT_WORDS=('why','what','wait','really','seriously','shocking','fight','argument','secret','unbelievable',
                 'kya','kyun','kaise','sach','jhagda','ladaai','larai','gussa','bilkul','nahi',
                 'کیا','کیوں','کیسے','لڑائی','جھگڑا','سچ','غصہ','نہیں','سنو','رکو','واقعی')

def sh(args, capture=False):
    return subprocess.run([str(x) for x in args],check=True,text=True,
        stdout=subprocess.PIPE if capture else None,stderr=subprocess.PIPE if capture else None)

def ffprobe_seconds(p):
    r=sh(['ffprobe','-v','error','-show_entries','format=duration',
          '-of','default=noprint_wrappers=1:nokey=1',p],capture=True)
    return float(r.stdout.strip())

def has_audio(p):
    r=sh(['ffprobe','-v','error','-select_streams','a:0','-show_entries','stream=index',
          '-of','csv=p=0',p],capture=True)
    return bool(r.stdout.strip())

def choose_source():
    if SOURCE_FILENAME.strip():
        p=INPUT/Path(SOURCE_FILENAME).name
        if not p.is_file(): raise FileNotFoundError(f'Upload {p.name} into INPUT first.')
        return p
    candidates=[p for p in INPUT.iterdir() if p.is_file() and p.suffix.lower() in VIDEO_EXTS]
    if not candidates: raise FileNotFoundError('Upload a video to Drive AI-Clip-Studio/INPUT first.')
    return max(candidates,key=lambda p:p.stat().st_mtime)

def read_plan(source):
    p=INPUT/'clip_plan.json'
    if not p.exists(): return {}
    data=json.loads(p.read_text(encoding='utf-8'))
    expected=Path(data.get('source_filename') or '').name
    if expected and expected!=source.name:
        raise ValueError(f'Clip plan belongs to {expected}; selected video is {source.name}. Choose matching source or remove old clip_plan.json.')
    return data

def read_custom(data,dur):
    rows=[]
    for entry in data.get('clips',[]):
        if entry.get('type') not in ('custom','transcript'): continue
        try: a,b=float(entry['start']),float(entry['end'])
        except (KeyError,ValueError,TypeError): continue
        if all(math.isfinite(v) for v in (a,b)) and 0<=a<b<=dur+0.1:
            rows.append({'start':round(a,3),'end':round(min(b,dur),3),
                         'origin':entry.get('type'),'score':None})
    return rows

def apply_transcript_edits(segs,data):
    edits=data.get('subtitle_edits') or []
    changed=0
    for edit in edits:
        try: a,b=float(edit['start']),float(edit['end']); text=str(edit['text']).strip()
        except (KeyError,ValueError,TypeError): continue
        if not text or len(text)>500: continue
        for s in segs:
            if abs(s['start']-a)<0.2 and abs(s['end']-b)<0.2:
                s['text']=text
                s['words']=[]  # use corrected whole segment as the subtitle line
                changed+=1
                break
    return changed

def transcribe_in_chunks(source,model,language='auto',chunk_minutes=8):
    if not has_audio(source): return [],['no audio track']
    chunks=WORK/'audio_chunks';chunks.mkdir(parents=True,exist_ok=True)
    for old in chunks.glob('part_*.wav'): old.unlink()
    # Segment small mono PCM files to prevent loading a full episode as one audio array.
    sh(['ffmpeg','-hide_banner','-loglevel','error','-y','-i',source,
        '-map','0:a:0','-vn','-ac','1','-ar','16000','-c:a','pcm_s16le',
        '-f','segment','-segment_time',str(60*chunk_minutes),'-reset_timestamps','1',
        chunks/'part_%04d.wav'])
    audios=sorted(chunks.glob('part_*.wav'))
    if not audios: return [],['no audio detected']
    all_segs=[]; offset=0; langs=[]
    for i,audio in enumerate(audios,1):
        print(f'Transcribing audio chunk {i}/{len(audios)}...')
        options=dict(beam_size=1,vad_filter=True,word_timestamps=True,condition_on_previous_text=False)
        if language in ('ur','en'): options['language']=language
        pieces,info=model.transcribe(str(audio),**options)
        langs.append(info.language)
        for piece in pieces:
            s={'start':round(offset+float(piece.start),3),'end':round(offset+float(piece.end),3),
               'text':piece.text.strip(), 'words':[]}
            s['words']=[{'start':round(offset+float(w.start),3),'end':round(offset+float(w.end),3),
                         'word':w.word} for w in (piece.words or []) if w.start is not None and w.end is not None]
            if s['text'] and s['end']>s['start']: all_segs.append(s)
        offset+=ffprobe_seconds(audio)
        audio.unlink()
    return all_segs,langs

def auto_candidates(segs,duration,n=3,target=35):
    target=min(duration,max(8,float(target)))
    spans=[]
    # Find local transcript windows by binary search instead of O(N²) scanning.
    segs=sorted(segs,key=lambda q:q['start'])
    starts=[q['start'] for q in segs]
    ends=[q['end'] for q in segs]
    for s in segs:
        center=(s['start']+s['end'])/2
        a=max(0,min(duration-target,center-target/2));b=min(duration,a+target)
        left=bisect_right(ends,a)
        right=bisect_left(starts,b)
        relevant=segs[left:right]
        joined=' '.join(q['text'] for q in relevant).lower()
        density=min(len(joined.split())/max(b-a,1),3)
        keywords=sum(joined.count(w) for w in HIGHLIGHT_WORDS)
        emphasis=joined.count('!')+.4*joined.count('?')
        talking=sum(max(0,min(b,q['end'])-max(a,q['start'])) for q in relevant)
        score=round(2*density+.8*min(keywords,7)+.45*min(emphasis,5)+talking/max(b-a,1),3)
        spans.append({'start':round(a,2),'end':round(b,2),'origin':'auto','score':score})
    spans.sort(key=lambda s:s['score'],reverse=True)
    chosen=[]
    for s in spans:
        if all(min(s['end'],q['end'])-max(s['start'],q['start'])<.15*min(s['end']-s['start'],q['end']-q['start']) for q in chosen):
            chosen.append(s)
            if len(chosen)>=n: break
    # Fall back to evenly-spaced suggestions if speech is absent or very sparse.
    if len(chosen)<n:
        for i in range(max(1,n*4)):
            center=(i+.5)*duration/(n*4)
            a=max(0,min(duration-target,center-target/2))
            s={'start':round(a,2),'end':round(min(duration,a+target),2),'origin':'fallback','score':0}
            if all(min(s['end'],q['end'])-max(s['start'],q['start'])<.15*min(s['end']-s['start'],q['end']-q['start']) for q in chosen): chosen.append(s)
            if len(chosen)>=n: break
    return sorted(chosen,key=lambda c:c['start'])

def srt_clock(t):
    ms=round(max(0,t)*1000)
    return f'{ms//3600000:02}:{(ms//60000)%60:02}:{(ms//1000)%60:02},{ms%1000:03}'

def save_srt(segs,a,b,dest):
    items=[]
    for seg in segs:
        if seg['end']<=a or seg['start']>=b: continue
        words=seg.get('words',[])
        if words:
            group=[]
            for word in words:
                wa,wb=word['start'],word['end']
                if wb<=a or wa>=b: continue
                group.append({'start':max(a,wa),'end':min(b,wb),'text':word['word']})
                if len(group)>=5 or group[-1]['end']-group[0]['start']>=2.4:
                    items.append((group[0]['start']-a,group[-1]['end']-a,''.join(x['text'] for x in group).strip()));group=[]
            if group: items.append((group[0]['start']-a,group[-1]['end']-a,''.join(x['text'] for x in group).strip()))
        else:
            items.append((max(seg['start'],a)-a,min(seg['end'],b)-a,seg['text'].strip()))
    lines=[]
    for st,en,txt in items:
        if not txt or en<=st:continue
        lines.append(f'{len(lines)+1}\n{srt_clock(st)} --> {srt_clock(en)}\n{txt}\n')
    dest.write_text('\n'.join(lines),encoding='utf-8')
    return len(lines)

def render(source,a,b,srt,outfile):
    fil=("[0:v]split=2[bg][fg];"
         "[bg]scale=720:1280:force_original_aspect_ratio=increase,crop=720:1280,boxblur=16:1[blur];"
         "[fg]scale=720:1280:force_original_aspect_ratio=decrease[front];"
         "[blur][front]overlay=(W-w)/2:(H-h)/2,format=yuv420p")
    if srt:
        # Arabic-script subtitles depend on FFmpeg libass and Noto fonts installed above.
        fil+=f",subtitles={srt}:force_style='FontName=Noto Naskh Arabic,FontSize=23,Outline=2,Shadow=0,Alignment=2,MarginV=130'"
    fil+='[v]'
    sh(['ffmpeg','-hide_banner','-loglevel','error','-y','-ss',str(a),'-i',source,
        '-t',str(b-a),'-filter_complex',fil,'-map','[v]','-map','0:a:0?',
        '-r','30','-c:v','libx264','-preset','veryfast','-crf','25',
        '-c:a','aac','-b:a','128k','-movflags','+faststart',outfile])


In [ ]:
#@title 4) Process footage, save transcript, create clips in Drive OUTPUT
start=time.time()
source=choose_source()
duration=ffprobe_seconds(source)
print('Video:',source.name,'| Duration:',round(duration/60,1),'minutes')
if duration>1800:
    print('LONG VIDEO: processing time and Colab session limits may be significant. This is not an unlimited-length guarantee.')
plan=read_plan(source)
plan_platform=plan.get('platform')
if plan_platform in PRESET_TARGETS:
    PLATFORM_PRESET=plan_platform
try:
    planned_target=float(plan.get('target_seconds',0))
except (TypeError,ValueError):
    planned_target=0
if 8<=planned_target<=180:
    TARGET_SECONDS=planned_target
elif AUTO_TARGET_SECONDS==0:
    TARGET_SECONDS=PRESET_TARGETS.get(PLATFORM_PRESET,35)
print('Target:',TARGET_SECONDS,'seconds | Platform:',PLATFORM_PRESET)
model=WhisperModel(MODEL_SIZE,device='cpu',compute_type='int8')
segs,langs=transcribe_in_chunks(source,model,SPEECH_LANGUAGE,CHUNK_MINUTES)
print('Recognized chunks:',len(langs),'| Language estimates:',langs[:12])
print('Transcript segments:',len(segs))
changed=apply_transcript_edits(segs,plan)
if changed: print('Subtitle corrections applied:',changed)

picked=[]
if MODE in ('auto','both'):
    picked.extend(auto_candidates(segs,duration,CLIPS_COUNT,TARGET_SECONDS))
if MODE in ('manual','both'):
    cuts=read_custom(plan,duration)
    if MODE=='manual' and not cuts:
        raise ValueError('Manual mode needs a clip_plan.json containing custom or transcript cuts. See GUIDE.html.')
    picked.extend(cuts)
if not picked: raise ValueError('No clips selected. Check the source video and clip plan.')

run_name=time.strftime('%Y%m%d_%H%M%S')
run_dir=OUTPUT/run_name;run_dir.mkdir(parents=True,exist_ok=True)
report={'source':source.name,'mode':MODE,'speech_language':SPEECH_LANGUAGE,
        'language_chunks':langs,'preset':PLATFORM_PRESET,'auto_target_seconds':TARGET_SECONDS,
        'subtitle_corrections':changed,'selection_method':'heuristic scoring (not guaranteed best moments)',
        'clips':[]}
(run_dir/'full_transcript.json').write_text(json.dumps({
    'version':2,'source_filename':source.name,'duration_seconds':duration,
    'detected_languages':langs,'segments':segs},ensure_ascii=False,indent=2),encoding='utf-8')
(run_dir/'full_transcript.txt').write_text('\n'.join(
    f"[{x['start']:.2f}-{x['end']:.2f}] {x['text']}" for x in segs),encoding='utf-8')
for ix,clip in enumerate(picked,1):
    a,b=clip['start'],clip['end']
    print(f'Clip {ix}/{len(picked)}: {a:.1f} to {b:.1f} ({clip["origin"]})')
    srt=WORK/f'clip_{ix:02}.srt'
    lines=save_srt(segs,a,b,srt)
    mp4_local=WORK/f'clip_{ix:02}.mp4'
    render(source,a,b,srt if lines else None,mp4_local)
    shutil.copy2(mp4_local,run_dir/mp4_local.name)
    if lines:shutil.copy2(srt,run_dir/srt.name)
    excerpt=' '.join(x['text'] for x in segs if x['end']>a and x['start']<b).strip()
    post_text=("Caption idea (please review): What do you think about this moment?\n\n"
        f"Dialogue: {excerpt[:350] if excerpt else '[No speech detected]'}\n\n"
        "Suggested tags: #Recap #Commentary #Shorts\n\n"
        "Check permission to reuse source footage before publishing.\n")
    (run_dir/f'clip_{ix:02}_post.txt').write_text(post_text,encoding='utf-8')
    report['clips'].append({'file':mp4_local.name,'start_sec':a,'end_sec':b,
                            'duration_sec':round(b-a,2),'origin':clip['origin'],
                            'heuristic_score':clip['score'],'caption_lines':lines})
(run_dir/'report.json').write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding='utf-8')
(run_dir/'README.txt').write_text('Review videos, Urdu/English subtitles, audio, and rights before manual posting. See GUIDE.html in starter package.\n',encoding='utf-8')
print('Complete:',len(picked),'clips ->',run_dir)
print('Open Google Drive on OPPO to review clips and full_transcript.json for text editing.')


## How to edit from words, not just times

1. Run once with `MODE = auto`. In the new `OUTPUT/<timestamp>/` folder download `full_transcript.json`.
2. Open `index.html` → **Text editor** → import that JSON. Search dialogue in Urdu or English. Correct the caption text in the editable boxes and/or tick lines for your desired clip.
3. Choose **Create clip from selected lines**; you can still edit the clip start/end times under **Custom clips**.
4. Tap **Download clip plan**. Upload `clip_plan.json` into the same Drive `INPUT` folder as your video.
5. Run this notebook again with `MODE = manual` (only chosen cuts) or `both` (auto + chosen cuts). Re-transcription occurs on rerun; text corrections are matched by timestamps.

**Note:** `index.html` works locally in Chrome as a planner. It does not upload or render media by itself. The transcript JSON can be large for long videos; importing it into a 4-GB phone may be slow. For best results, start with a 3–5 minute test, then increase input size gradually. Urdu/English recognition is imperfect; check subtitles. See **GUIDE.html**.
